<a href="https://colab.research.google.com/github/osacr6/SC707_Proyecto_G6/blob/Daniel/SC707_Proyecto_G6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Perfiles económicos de los cantones de Costa Rica según sus pymes activas
## Proyecto final · Avance 2 — Grupo 6 · SC-707 Big Data · Universidad Fidélitas

**Pregunta:** ¿qué cantones pueden identificarse con K-means, considerando la cantidad, las actividades económicas y la evolución del 2020 al 2026 de sus pymes y qué los diferencia?

**Datos:** cada registro es una pyme en un año y un cantón (lista de pymes activas del MEIC, enero de 2020 a 2026), más la población de cada cantón.

**Subir todos los archivos en `MyDrive/sc-707-pymes`**.

# 1 - Datos

In [10]:
from google.colab import drive
drive.mount('/content/drive')


import os, re, unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Subir archivos a Drive
DATA_DIR = BASE_DIR = '/content/drive/MyDrive/sc-707-pymes'
AÑOS = range(2020, 2027)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.1 Unir archivos

- Cada año trae columnas con otros nombres, en otro orden o faltan.
- Se conservan las que existen en todos los años

In [12]:
REGLAS = [
  ('IDENTIFICACI', 'ID'),
  ('TAMA', 'TAMANO'),
  ('PROVINCIA', 'PROVINCIA'),
  ('CANT', 'CANTON'),
  ('REGI', 'REGION'),
  ('ACTIVIDAD CIIU', 'CIIU'),
  ('ACTIVIDAD PRIMARIA', 'CIIU')
]

def _norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', s).strip().upper()

def renombrar(cols):
    nuevo = {}
    for c in cols:
        for prefijo, nombre in REGLAS:
            if _norm(c).startswith(prefijo) and nombre not in nuevo.values():
                nuevo[c] = nombre
                break
    return nuevo

estructura, partes = [], []
for anio in AÑOS:
    crudo = pd.read_csv(os.path.join(DATA_DIR, f'{anio}-01.csv'), encoding='latin-1', dtype=str)
    crudo = crudo.loc[:, ~crudo.columns.str.startswith('Unnamed')].dropna(how='all')
    mapa = renombrar(crudo.columns)
    t = crudo.rename(columns=mapa)[list(mapa.values())].copy()
    t['ANIO'] = anio
    estructura.append({'Año': anio, 'Registros': len(t), 'Columnas': crudo.shape[1]})
    partes.append(t)

serie = pd.concat(partes, ignore_index=True)
for c in ['ID', 'TAMANO', 'PROVINCIA', 'CANTON', 'REGION', 'CIIU']:
    serie[c] = serie[c].astype('string').str.strip()

estructura = pd.DataFrame(estructura)

estructura

,Año,Registros,Columnas
0,2020,19923,12
1,2021,24038,11
2,2022,25691,11
3,2023,21644,12
4,2024,25606,12
5,2025,25661,9
6,2026,31757,13
